# DrugReview — one shared split for every downstream notebook

Creates `00_split/data/DrugReview_split.csv` **once**. Folders 02–07, 13 and ablation/A0_Matched_holistic_control read this file and no longer
compute their own split, so text-only and text + metadata runs, and all three targets, are compared on
exactly the same reviews.

* The unit of splitting is the **narrative** (cleaned review text); a narrative never crosses splits.
* All 500 human-annotated reviews, and every other row that shares a narrative with one of them, are
  reserved as `split = "audit"`: never used for training, validation, TF-IDF or vocabulary fitting.
  They are predicted separately and reported separately from the ordinary test set.
* The remaining narratives are split 60 / 20 / 20 with the same procedure and seed as before.
* The split reads only the cleaned corpus (review text) and the audit key. It never sees an annotation, so it
  stays fixed when the labels are replaced.

# 1. Configuration

In [ ]:
import os, re, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

if Path("/content").is_dir() and not os.environ.get("DRUGREVIEW_ROOT"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
import os
from pathlib import Path

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents,
                 Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

PROJECT_ROOT = resolve_project_root()

DRIVE_PROJECT_PATH = str(PROJECT_ROOT / "00_split")
DATA_DIR   = Path(DRIVE_PROJECT_PATH) / "data"                  # this step keeps its input (audit key) and output (split) here
CORPUS_FILE = (Path(DRIVE_PROJECT_PATH) / ".." / "00_labelling" / "output" /
               "drugscom_labelling_wcge50_ok_dedup_text_cond_ingCLEAN.csv")   # written by DrugReview_CleanComments
AUDIT_KEY  = DATA_DIR / "human_validation_sample_key_500.csv"      # the 500 human-annotated reviews
SPLIT_FILE = DATA_DIR / "DrugReview_split.csv"                     # <- written here

ID_COLUMN    = "uniqueID"
TEXT_COLUMN  = "review"
RANDOM_STATE = 42

CLASS_NAMES  = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]

In [ ]:
# Google Drive is mounted by the setup cell when needed.


# 2. Load the corpus and validate the cohort

In [ ]:
# The split depends only on the review text and the audit key - never on any annotation.
df = pd.read_csv(CORPUS_FILE)
assert "review_raw" in df.columns, "Rerun DrugReview_CleanComments first: the split must hash the decoded review text the models read."
print(f"Rows: {len(df)}")

def md5_file(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

CORPUS_FILE_MD5 = md5_file(CORPUS_FILE)
print("Corpus file md5:", CORPUS_FILE_MD5)

# One eligible cohort for every target: stop if anything would make a notebook drop rows.
assert df[ID_COLUMN].is_unique, "uniqueID is not unique"
assert df[TEXT_COLUMN].notna().all() and (df[TEXT_COLUMN].astype(str).str.strip() != "").all(), "empty reviews"
assert df["rating"].notna().all(), "missing ratings"
print("Cohort checks passed: unique IDs, no empty review, no missing rating.")

# 3. Narrative groups

In [ ]:
def clean_text(text: str) -> str:            # identical to the downstream notebooks
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

_clean = df[TEXT_COLUMN].apply(clean_text)
df["narrative_group"] = _clean.map(lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()[:16])
assert df.groupby("narrative_group")[TEXT_COLUMN].agg(lambda s: s.map(clean_text).nunique()).max() == 1, "hash collision"

_size = df["narrative_group"].map(df["narrative_group"].value_counts())
print(f"Narratives: {df['narrative_group'].nunique()} | rows sharing a narrative with another row: "
      f"{int((_size > 1).sum())} ({(_size > 1).mean():.1%}) | largest group: {int(_size.max())}")

# 4. Human-audit reviews

In [ ]:
key = pd.read_csv(AUDIT_KEY)
print("Audit key columns:", key.columns.tolist(), "| rows:", len(key))

if ID_COLUMN in key.columns:
    audit_ids = key[ID_COLUMN].tolist()
elif "source_row_id" in key.columns:          # row position in the label file
    audit_ids = df.iloc[key["source_row_id"].astype(int).to_numpy()][ID_COLUMN].tolist()
else:
    raise KeyError("The audit key needs a 'uniqueID' or a 'source_row_id' column.")

# If the key carries the review text, it must be the same text as the matched corpus row.
_text_col = next((c for c in ["review", "text", "review_text"] if c in key.columns), None)
if _text_col is not None:
    # Every key row must be the same text as the corpus row it points to (original or decoded text) - row by row.
    _sel = df.set_index(ID_COLUMN).loc[audit_ids]
    _key_text = key[_text_col].map(clean_text).to_numpy()
    _match = (_sel["review_raw"].map(clean_text).to_numpy() == _key_text) | (_sel[TEXT_COLUMN].map(clean_text).to_numpy() == _key_text)
    print(f"Audit key rows whose text is the matched corpus row: {_match.sum()} of {len(key)}")
    if not _match.all():
        print(key.loc[~_match].head(10).to_string())
    assert _match.all(), (f"{(~_match).sum()} audit key row(s) point to a different review. Resolve them one by one "
                          "(a positional source_row_id only works for the corpus file it was made from).")
else:
    print("NOTE: the audit key has no text column, so its rows cannot be checked against the corpus text.")

assert len(set(audit_ids)) == len(key), "duplicate reviews in the audit key"
df["is_human_annotated"] = df[ID_COLUMN].isin(audit_ids)
df["is_audit_group"] = df["narrative_group"].isin(df.loc[df["is_human_annotated"], "narrative_group"])

print(f"Human-annotated reviews: {int(df['is_human_annotated'].sum())} | rows reserved with them: {int(df['is_audit_group'].sum())}")
assert int(df["is_human_annotated"].sum()) == len(key)

# 5. Split

In [ ]:
df["split"] = ""
df.loc[df["is_audit_group"], "split"] = "audit"

rest = df[~df["is_audit_group"]]
y_dummy = np.zeros(len(rest))

# 80/20 -> trainval / test, then 75/25 -> train / val  (about 60/20/20), by narrative
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
trainval_pos, test_pos = next(gss_test.split(rest, y_dummy, groups=rest["narrative_group"]))
trainval = rest.iloc[trainval_pos]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_pos, val_pos = next(gss_val.split(trainval, np.zeros(len(trainval)), groups=trainval["narrative_group"]))

df.loc[rest.index[test_pos], "split"] = "test"
df.loc[trainval.index[train_pos], "split"] = "train"
df.loc[trainval.index[val_pos], "split"] = "val"

assert (df["split"] != "").all()
assert (df.groupby("narrative_group")["split"].nunique() == 1).all(), "a narrative crosses splits"
assert not df.loc[df["is_audit_group"], "split"].isin(["train", "val", "test"]).any()
print(df["split"].value_counts().reindex(["train", "val", "test", "audit"]))
print(df["split"].value_counts(normalize=True).reindex(["train", "val", "test", "audit"]).round(4))

## 5.1 Class balance of the rating target in every split

In [ ]:
def rating_to_sentiment_5(r):
    r = int(r)
    if r <= 2:  return "VERY_NEGATIVE"
    if r <= 4:  return "NEGATIVE"
    if r <= 6:  return "NEUTRAL"
    if r <= 8:  return "POSITIVE"
    return "VERY_POSITIVE"

targets = pd.DataFrame({"rating": df["rating"].map(rating_to_sentiment_5)})
for _t in targets.columns:
    tab = pd.crosstab(targets[_t], df["split"]).reindex(CLASS_NAMES)[["train", "val", "test", "audit"]]
    print(f"\n=== {_t} ===")
    print(pd.concat([tab, (tab / tab.sum() * 100).round(1).add_suffix(" %")], axis=1).to_string())

# 6. Save the manifest

In [ ]:
manifest = df[[ID_COLUMN, "narrative_group", "split", "is_human_annotated", "is_audit_group"]].copy()
manifest.to_csv(SPLIT_FILE, index=False)

SPLIT_FINGERPRINT = hashlib.md5(
    "|".join(f"{u}:{s}" for u, s in sorted(zip(manifest[ID_COLUMN].astype(str), manifest["split"]))).encode()
).hexdigest()
print(f"Saved -> {SPLIT_FILE}  ({len(manifest)} rows)")
print("Split fingerprint :", SPLIT_FINGERPRINT, " <- every downstream notebook prints the same value")
print("Corpus file md5   :", CORPUS_FILE_MD5)

with open(DATA_DIR / "DrugReview_split_info.json", "w") as f:
    json.dump({"split_fingerprint": SPLIT_FINGERPRINT, "corpus_file_md5": CORPUS_FILE_MD5,
               "n_rows": int(len(manifest)), "counts": manifest["split"].value_counts().to_dict(),
               "n_human_annotated": int(manifest["is_human_annotated"].sum()),
               "n_audit_group": int(manifest["is_audit_group"].sum()),
               "random_state": RANDOM_STATE}, f, indent=2)